# Proyecto I — Validador de direcciones IPv4

## Contexto y reglas

El proyecto utiliza dos autómatas finitos para analizar
direcciones IPv4. En este apartado nos enfocaremos solomaente en el AFD.

**AFD (L1):** Valida cuatro octetos entre 0 y 255,
separados por puntos y sin ceros iniciales.

Alfabeto: {0,1,2,3,4,5,6,7,8,9,.}


### Definición del AFD

Se utilizan los estados A0–A12, B0–B12, C0–C12 y D0–D12 para representar los cuatro octetos. El estado T funciona como estado trampa.

Los estados recuerdan la cantidad y el rango de los dígitos leídos, permitiendo verificar que cada octeto esté entre 0 y 255.

Los estados de aceptación pertenecen al cuarto octeto y solo aceptan si se ha consumido toda la entrada.


In [1]:
# Definicion del AFD
alfabeto = {"0", "1", "2", "3", "4",
            "5", "6", "7", "8", "9", "."}
AFD = {
    "estados": set(),
    "inicial": "A0",
    "alfabeto": alfabeto,
    "aceptacion": set(),
    "delta": {}
}
# Creamos los estados A, B, C y D
for letra in ["A", "B", "C", "D"]:
    for numero in range(13):
        AFD["estados"].add(letra + str(numero))
AFD["estados"].add("T")
# Estados finales del cuarto octeto
for numero in range(1, 13):
    AFD["aceptacion"].add("D" + str(numero))
print("Estados:", len(AFD["estados"]))
print("Inicial:", AFD["inicial"])
print("Finales:", sorted(AFD["aceptacion"]))

Estados: 53
Inicial: A0
Finales: ['D1', 'D10', 'D11', 'D12', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9']


### Función de transición del AFD

La función de transición determina el siguiente estado a partir del estado actual y el símbolo leído.

$\delta_1:Q_1\times\Sigma_1\rightarrow Q_1$

Se utiliza una tabla de transiciones para procesar cada octeto. Las mismas reglas se aplican a los cuatro octetos y cualquier transición inválida conduce al estado trampa T.

In [2]:
#  Definicion de las transiciones del AFD
tabla = {
    0: {"0": 1, "1": 2, "2": 3},
    1: {},
    2: {},
    3: {"5": 7},
    4: {},
    5: {},
    6: {},
    7: {},
    8: {},
    9: {},
    10: {},
    11: {},
    12: {}
}
# A0: primer digito entre 3 y 9
for d in "3456789":
    tabla[0][d] = 4
# A2: valores 10-19
for d in "0123456789":
    tabla[2][d] = 5
# A3: valores 20-24
for d in "01234":
    tabla[3][d] = 6
# A3: valores 26-29
for d in "6789":
    tabla[3][d] = 8
# Segundos y terceros digitos
for d in "0123456789":
    tabla[4][d] = 9
    tabla[5][d] = 10
    tabla[6][d] = 11
# A7: valores 250-255
for d in "012345":
    tabla[7][d] = 12
# Transiciones de los cuatro octetos
letras = ["A", "B", "C", "D"]
for i in range(4):
    letra = letras[i]
    for numero in range(13):
        estado = letra + str(numero)
        for simbolo in alfabeto:
            destino = "T"
            if simbolo == ".":
                if numero != 0 and i < 3:
                    destino = letras[i + 1] + "0"
            elif simbolo in tabla[numero]:
                destino = letra + str(tabla[numero][simbolo])
            AFD["delta"][(estado, simbolo)] = destino
# Estado trampa
for simbolo in alfabeto:
    AFD["delta"][("T", simbolo)] = "T"
print("Transiciones creadas:", len(AFD["delta"]))

Transiciones creadas: 583


## Simulador del AFD

Se implementa una función que recibe una cadena y recorre sus símbolos uno por uno.

El simulador comienza en el estado inicial A0 y mantiene un solo estado actual durante toda la lectura. Para cada símbolo consulta el diccionario de transiciones.

Al terminar, acepta la cadena únicamente si el estado alcanzado pertenece al conjunto de estados finales.


In [3]:
# Simulador del AFD
def ejecutar(AFD, cadena):
    estado = AFD["inicial"]
    for simbolo in cadena:
        if simbolo not in AFD["alfabeto"]:
            print("Error: simbolo fuera del alfabeto")
            return False
        estado = AFD["delta"][(estado, simbolo)]
    return estado in AFD["aceptacion"]
# Probamos una cadena
cadena = "192.168.1.10"
print("La cadena", cadena, "esta:")
if ejecutar(AFD, cadena):
    print("ACEPTADA")
else:
    print("RECHAZADA")

La cadena 192.168.1.10 esta:
ACEPTADA


## Simulador del AFD
En este apartado se realizaran pruebas con diferentes tipos de cadena y de esa manera lograr ver si las acepta o rechaza 

In [4]:
#Pruebas del AFD
pruebas_afd = [
    "10.0.0.1",
    "172.16.0.1",
    "192.168.1.10",
    "224.0.0.1",
    "255.255.255.255",
    "256.1.1.1",
    "192.168.1",
    "192..1.10",
    "01.2.3.4",
    "192.168.1.999",
    "",
    "192.168.A.1"
]
print("RESULTADOS DEL AFD")
for cadena in pruebas_afd:
    print("\nCadena:", repr(cadena))
    if any(s not in AFD["alfabeto"] for s in cadena):
        print("FUERA DEL ALFABETO")
    elif ejecutar(AFD, cadena):
        print("ACEPTADA")
    else:
        print("RECHAZADA")

RESULTADOS DEL AFD

Cadena: '10.0.0.1'
ACEPTADA

Cadena: '172.16.0.1'
ACEPTADA

Cadena: '192.168.1.10'
ACEPTADA

Cadena: '224.0.0.1'
ACEPTADA

Cadena: '255.255.255.255'
ACEPTADA

Cadena: '256.1.1.1'
RECHAZADA

Cadena: '192.168.1'
RECHAZADA

Cadena: '192..1.10'
RECHAZADA

Cadena: '01.2.3.4'
RECHAZADA

Cadena: '192.168.1.999'
RECHAZADA

Cadena: ''
RECHAZADA

Cadena: '192.168.A.1'
FUERA DEL ALFABETO
